# ULDM Spherical Harmonic Analysis Pipeline

This notebook serves as the **entry point** for the ultralight dark matter (ULDM) wavefunction analysis pipeline. It orchestrates the computation of spherical harmonic coefficients $a_{lm}(r, t)$ from PyUL simulation outputs.

## Pipeline Overview

The complete analysis workflow spans **three notebooks**:

| Notebook | Purpose | Key Outputs |
|----------|---------|-------------|
| **Main.ipynb** (this file) | Generate $a_{lm}(r,t)$ from simulation data | `PyUL_psi_alm_*.h5` |
| **f_nl_python.ipynb** | Compute radial eigenfunctions $f_{nl}(r)$ | `eigs_*.h5` |
| **compute_c_nlm.ipynb** | Compute time-dependent coefficients $c_{nlm}(t)$ | `c_nlm_*.h5`, figures |

## Workflow Diagram

```
┌─────────────────────────────────────────────────────────────────────────┐
│  Main.ipynb                                                              │
│  ┌──────────────┐    ┌──────────────┐    ┌──────────────┐               │
│  │ Step 1       │───▶│ Step 2       │───▶│ Step 3       │               │
│  │ Prepare r₀   │    │ Run compute  │    │ Merge into   │               │
│  │ segments     │    │ _coeff*.py   │    │ a_lm HDF5    │               │
│  └──────────────┘    └──────────────┘    └──────────────┘               │
└─────────────────────────────────────────────────────────────────────────┘
                              ▼
┌─────────────────────────────────────────────────────────────────────────┐
│  f_nl_python.ipynb (Steps 1–5)                                          │
│  Compute eigenfunctions f_{nl}(r) ──▶ eigs_*.h5                         │
└─────────────────────────────────────────────────────────────────────────┘
                              ▼
┌─────────────────────────────────────────────────────────────────────────┐
│  compute_c_nlm.ipynb                                                     │
│  Compute c_{nlm}(t) via radial integration ──▶ c_nlm_*.h5 + figures      │
└─────────────────────────────────────────────────────────────────────────┘
```

## Prerequisites

- **PyUL simulation output**: The `3WfnRS/` (or `3Wfn/`) folder containing wavefunction snapshots
- **Python environment**: `numpy`, `scipy`, `pyshtools`, `h5py`, `matplotlib`, plus `Eig_ULDM_packages`
- **Computational resources**: Steps 1–2 are CPU-intensive; PBS/cluster usage is recommended

## Key Mathematical Background

The ULDM wavefunction $\psi(\mathbf{r}, t)$ is expanded in spherical harmonics:

$$\psi(\mathbf{r}, t) = \sum_{\ell=0}^{\ell_{\max}} \sum_{m=-\ell}^{\ell} a_{\ell m}(r, t) \, Y_\ell^m(\theta, \phi)$$

This notebook computes the radial coefficients $a_{\ell m}(r, t)$ by projecting the 3D wavefunction onto the spherical harmonic basis at each radial shell and time step.

## Step 1: Prepare $r_0$ Segments and Compute Scripts

This cell prepares the infrastructure for parallel spherical harmonic decomposition:

1. **Build `r_0_values.npy`**: A radial grid from `r_min` to `r_max` with `resolution + 1` points
2. **Split into segments**: Divide the radial grid into `n_segments` files (`r0_segment1.npy`, etc.)
3. **Generate compute scripts**: Create `compute_coeff1.py` ... `compute_coeffN.py` from the template

**User parameters** (edit before running):
- `work_dir`: Absolute path to the PyUL `Outputs/` directory
- `resolution`: Number of grid cells (must match simulation resolution)
- `box_length`: Full simulation box length in kpc
- `n_segments`: Number of parallel jobs (match your available compute nodes)

**⚠️ Important**: Edit `compute_coeff.py` (template) first to set physical parameters matching your PyUL run.

In [1]:
import os
import numpy as np

base_dir = os.getcwd()  # notebook directory (template location)

# ------------------------ configurable parameters ------------------------
# set absolute path to Outputs; leave empty to keep current directory
work_dir = "/data/maudlin/alanz/Binary_soliton/128_6000_1000_doubleBoxsize_2/20260130_221325@128/Outputs"  
if work_dir:
    os.chdir(work_dir)
resolution = 256   # number of grid cells along one axis (≠ number of sample points)
r_min      = 0  # starting radius
box_length = 2.5*2  # full box length
r_max      = box_length/2  # ending radius (box_length / 2, etc.)
n_segments = 4      # number of r0 segments (must match number of compute scripts)

# Template for compute_coeff scripts (edit this file first)
template_path = os.path.join(base_dir, "compute_coeff.py")
output_script_dir = os.path.dirname(template_path)
os.makedirs(output_script_dir, exist_ok=True)
# ------------------------------------------------------------------------

# 1. build the full r_0 array and save
n_points = resolution + 1                       # include the endpoint
r_0_values = np.linspace(r_min, r_max, n_points, endpoint=True)
r_0_values[0]= 1e-5  # ensure the first value is exactly 1e-5 (avoid zero radius)
np.save("r_0_values.npy", r_0_values)           # full array on disk

# 2. split into N segments, distributing any remainder to the front segments
base_len, remainder = divmod(n_points, n_segments)
lengths = [base_len + (1 if i < remainder else 0) for i in range(n_segments)]

start = 0
segments = []
for seg_len in lengths:
    segments.append(r_0_values[start : start + seg_len])
    start += seg_len

# 2b. save each segment as its own .npy file  -----------------------------
for idx, seg in enumerate(segments, start=1):
    np.save(f"r0_segment{idx}.npy", seg)        # e.g. r0_segment1.npy
print(f"Saved {n_segments} segments as r0_segment1.npy ~ r0_segment{n_segments}.npy")
# ------------------------------------------------------------------------

# 3. recombine to verify correctness (optional)
r_0_recombined = np.concatenate(segments)
print(f"Array length matches: {len(r_0_values) == len(r_0_recombined)}")
print(f"Arrays are identical: {np.array_equal(r_0_values, r_0_recombined)}")

# 4. generate compute_coeff*.py scripts from the template
with open(template_path, "r") as f:
    template_text = f.read()
needle = 'r_0_values = np.load("r_0_values.npy") # Load r0 values from file'
if needle not in template_text:
    raise ValueError(f"Template line not found: {needle}")

for idx in range(1, n_segments + 1):
    updated = template_text.replace(
        needle,
        f'r_0_values = np.load("r0_segment{idx}.npy") # Load r0 values from file',
        1,
    )
    out_path = os.path.join(output_script_dir, f"compute_coeff{idx}.py")
    with open(out_path, "w") as f:
        f.write(updated)
print(f"Generated compute_coeff1.py ~ compute_coeff{n_segments}.py in {output_script_dir}")

Saved 4 segments as r0_segment1.npy ~ r0_segment4.npy
Array length matches: True
Arrays are identical: True
Generated compute_coeff1.py ~ compute_coeff4.py in /data/maudlin/alanz/Test/Outputs


## Step 2: Run `compute_coeff*.py` Scripts (External)

This step is executed **outside the notebook** (PBS cluster or local terminal). The scripts perform the computationally intensive spherical harmonic decomposition.

### PBS / Cluster Usage (Recommended)

1. Edit `submit_all_jobs_example.sh`:
   - Set `WORK_DIR` to your `Outputs/` path
   - Set `ENV_NAME` to your conda environment
   - Set `JOB_COUNT` to match `n_segments` from Step 1
   - Optionally set `NODE_LIST` to pin jobs to specific nodes

2. Submit jobs:
   ```bash
   bash submit_all_jobs_example.sh
   ```

3. Monitor progress via log files in `WORK_DIR/logs/`

### Local Usage

```bash
cd /path/to/Outputs
conda activate your_env
python compute_coeff1.py
python compute_coeff2.py
# ... etc.
```

**Output**: JSON files in `alm_coefficients/` subdirectory, organized by `r0` value and `l_max`.

**⚠️ Note**: This step can take hours to days depending on resolution and time range. Ensure adequate computational resources before proceeding.

## Step 3: Merge Raw Data into $a_{lm}$ HDF5 File

After Step 2 completes, this cell consolidates the JSON coefficient files into a single HDF5 file suitable for downstream analysis.

**Function**: `create_sh_hdf5()` from `Eig_ULDM_packages.alm_utils`

**⚠️ User Configuration Required**:
- `work_dir`: **Change this** to your PyUL simulation output directory
- `output_path`: Name of the output HDF5 file
- `start_index`, `end_index`: Time snapshot range to include
- `n_r`: Number of radial points (must match Step 1 `resolution + 1`)
- `l_max`: Maximum angular momentum quantum number
- `r_values`: Radial grid in physical units (kpc)
- `file_prefix`: Path pattern for wavefunction files (`3WfnRS/P3R_#` or `3Wfn/P3D_#`)

**Output**: `PyUL_psi_alm_*.h5` containing:
- `a_lm(r, t)` arrays for each $(\ell, m)$ pair
- Time and radial grid metadata

**Tip**: Use `analyze_h5_structure(output_path)` to inspect the file structure after creation.

In [ ]:
import numpy as np
import os
from Eig_ULDM_packages.alm_utils import analyze_h5_structure, create_sh_hdf5

# ============================================================================
# USER CONFIGURATION — Modify these paths and parameters for your simulation
# ============================================================================
work_dir = "/path/to/your/simulation/Outputs"  # <-- CHANGE THIS to your PyUL output directory
# Example: work_dir = "/data/user/Stone_Skipping/128_0.02_3000_500/20251016_113439@128/Outputs"

if work_dir:
    os.chdir(work_dir)

# Output file name for the a_lm HDF5 data
output_path = "PyUL_psi_alm.h5"

# Time snapshot range (adjust based on your simulation)
start_index = 500
end_index = 1000

# Radial grid parameters
n_r = 257
l_max = 10
num_points = 300
r_min = 0       # starting radius (kpc)
r_max = 2.5     # ending radius (kpc) — typically box_length / 2

r_values = np.linspace(r_min, r_max, n_r, endpoint=True)
r_values[0] = 1e-5  # avoid zero radius for numerical stability

# Wavefunction file pattern in your simulation output
file_prefix = "3WfnRS/P3R_#"  # or "3Wfn/P3D_#" for non-resampled data

# Number of parallel workers
n_workers = 8
# ============================================================================

create_sh_hdf5(
    output_path=output_path,
    start_index=start_index,
    end_index=end_index,
    n_r=n_r,
    l_max=l_max,
    num_points=num_points,
    r_values=r_values,
    file_prefix=file_prefix,
    n_workers=n_workers,
)

# Optionally inspect the file structure
# analyze_h5_structure(output_path)

Creating HDF5 file: PyUL_psi_alm_0.02_mass170_256.h5
Processing 501 time points with 257 radial points
Using l_max = 10, resulting in 121 (l,m) combinations
Using 8 worker processes


Processing time points: 100%|██████████| 501/501 [01:13<00:00,  6.80step/s]


HDF5 file creation complete.
=== HDF5 File Structure ===

Group: alm_000000
  Dataset: alm_000000/counts
  └─ Shape: (257, 121), Dtype: float64
  Dataset: alm_000000/ylm
  └─ Shape: (257, 121), Dtype: complex128
Group: alm_000001
  Dataset: alm_000001/counts
  └─ Shape: (257, 121), Dtype: float64
  Dataset: alm_000001/ylm
  └─ Shape: (257, 121), Dtype: complex128
Group: alm_000002
  Dataset: alm_000002/counts
  └─ Shape: (257, 121), Dtype: float64
  Dataset: alm_000002/ylm
  └─ Shape: (257, 121), Dtype: complex128
Group: alm_000003
  Dataset: alm_000003/counts
  └─ Shape: (257, 121), Dtype: float64
  Dataset: alm_000003/ylm
  └─ Shape: (257, 121), Dtype: complex128
Group: alm_000004
  Dataset: alm_000004/counts
  └─ Shape: (257, 121), Dtype: float64
  Dataset: alm_000004/ylm
  └─ Shape: (257, 121), Dtype: complex128
Group: alm_000005
  Dataset: alm_000005/counts
  └─ Shape: (257, 121), Dtype: float64
  Dataset: alm_000005/ylm
  └─ Shape: (257, 121), Dtype: complex128
Group: alm_000006


## Step 4: Compute Radial Eigenfunctions — Go to `f_nl_python.ipynb`

With the $a_{lm}(r, t)$ data ready, the next step is to compute the radial eigenfunctions $f_{nl}(r)$ that form the basis for the eigenmode expansion.

### Action Required

Open **`f_nl_python.ipynb`** and run **Steps 1–5** to:

1. Load the soliton density profile
2. Construct the gravitational potential $\Phi(r)$
3. Solve the radial Schrödinger equation for each $\ell$
4. Visualize and save the eigenfunctions to `eigs_*.h5`

**Output**: `eigs_mass_*.h5` file containing $f_{nl}(r)$ and eigenvalues $E_{nl}$

### Additional Features in `f_nl_python.ipynb`

Steps 6–9 in that notebook provide **wavefunction reconstruction** capabilities:
- Build initial 3D wavefunctions from $c_{nlm}$ coefficients
- Apply center-of-mass and velocity corrections
- Compute mode mass fractions

These features require the $c_{nlm}(t)$ file from the next step, so return here after completing Steps 1–5.

## Step 5: Compute $c_{nlm}(t)$ Coefficients — Go to `compute_c_nlm.ipynb`

With both $a_{lm}(r, t)$ (from Step 3) and $f_{nl}(r)$ (from Step 4) available, you can now compute the time-dependent expansion coefficients.

### Action Required

Open **`compute_c_nlm.ipynb`** and run all cells sequentially to:

1. Load eigenfunctions and spherical harmonic data
2. Compute $c_{nlm}(t)$ via radial integration:
   $$c_{nlm}(t) = \int_0^{r_{\max}} f_{nl}(r) \, a_{lm}(r,t) \, r^2 \, dr$$
3. Generate time-interpolated dense sampling (optional)
4. Perform spectral analysis and visualization
5. Export publication-ready figures

**Outputs**:
- `c_nlm_*.h5`: Time-dependent coefficients
- `c_nlm_*_dense_from_file.h5`: Interpolated dense time series
- `PRD_cnlm_*.eps`: Publication-ready figures

### After Completion

You may return to `f_nl_python.ipynb` Steps 6–9 if you wish to:
- Reconstruct initial wavefunctions with specific mode compositions
- Apply deboosting for center-of-mass and velocity corrections
- Validate mode mass fractions

---

**Congratulations!** You have completed the ULDM eigenstate analysis pipeline.